# EDA inicial do LEN-small

Este notebook abre a tabela produzida por `len-summary`. Os vetores LaBSE não são carregados nesta etapa.

## Preparar as ferramentas e localizar os arquivos

Importa as bibliotecas usadas no notebook e informa onde estão: a pasta original do LEN e a tabela-resumo criada na primeira leitura.

In [ ]:
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'small_encoder_final'
SUMMARY_PATH = PROJECT_ROOT / 'outputs' / 'len_small_summary.csv'
sns.set_theme(style='whitegrid')

## Abrir a tabela de resumo dos grafos

A tabela possui uma linha por grafo do LEN. Se ela ainda não existir, esta célula executa o leitor `len-summary`, que percorre todos os JSONs e gera o arquivo. 
Depois, ela abre a tabela na variável `graphs` e mostra as primeiras linhas para conferência. 
Na primeira execução sem a tabela, a leitura completa pode levar alguns minutos.

In [ ]:
if not SUMMARY_PATH.exists():
    subprocess.run([
        'len-summary', '--data-dir', str(DATA_DIR), '--output', str(SUMMARY_PATH)
    ], check=True)

graphs = pd.read_csv(SUMMARY_PATH)
graphs.head()

## Comparar campanhas e não campanhas

Agrupa os grafos pelas duas classes do dataset: `campaign` e `noncampaign`. 
Para cada grupo, ela apresenta quantos grafos existem e valores típicos de tamanho, duração e cobertura temporal. 
Esses números ajudam a perceber diferenças que podem influenciar os modelos mais adiante.

In [ ]:
graphs.groupby('label').agg(
    graphs=('file', 'count'),
    nodes_mean=('nodes', 'mean'),
    nodes_median=('nodes', 'median'),
    edges_mean=('edges', 'mean'),
    duration_hours_median=('duration_hours', 'median'),
    timestamp_coverage_min=('timestamp_coverage', 'min'),
).round(3)

## Visualizar tamanho e conexão dos grafos

O primeiro gráfico mostra a relação entre número de nós e número de arestas de cada grafo. As escalas são logarítmicas para caberem grafos pequenos e grandes na mesma figura. 
O segundo compara, entre as classes, a fração de nós presente no maior componente conectado; ele ajuda a entender se a rede está concentrada ou fragmentada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=graphs, x='nodes', y='edges', hue='label', ax=axes[0])
axes[0].set(xscale='log', yscale='log', title='Tamanho dos grafos')
sns.boxplot(data=graphs, x='label', y='largest_weak_component_fraction', ax=axes[1])
axes[1].set(title='Fração do maior componente fracamente conectado')
plt.tight_layout()

## Procurar problemas de qualidade dos dados

Checagem de segurança para a pesquisa. Ela lista somente os grafos com identificadores de nó duplicados, arestas ligadas a nós inexistentes ou timestamps ausentes. 
O resultado vazio é bom: significa que nenhum desses problemas foi encontrado pelos critérios verificados.

In [ ]:
integrity_columns = [
    'file', 'duplicate_node_ids', 'missing_edge_endpoints',
    'timestamps_missing', 'timestamp_coverage',
]
graphs.loc[
    (graphs['duplicate_node_ids'] > 0)
    | (graphs['missing_edge_endpoints'] > 0)
    | (graphs['timestamp_coverage'] < 1),
    integrity_columns,
]